# CS689 Term Project – Update 2: Part 4 – Dimensions
**Student:** Medha Prodduturi  
**Topic:** Audience Consumption Evolution: Traditional Media vs. Digital Streaming Platforms

---

## Business Questions (from Update 1)
1. Which traditional media platforms observed decline/impact as digital platforms grew, and which modern streaming platforms show most engagement over time?
2. When did the shift from traditional to digital media accelerate most significantly?
3. Which audience segments (age/income/location) drive the shift to digital platforms?
4. How does engagement differ between traditional and digital streaming (time spent/retention/frequency)?
5. What major factors influence users to switch from traditional to digital media?

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load all data sources
streaming_service   = pd.read_csv('Data Sources copy/streaming_service.csv')
web_traffic         = pd.read_csv('Data Sources copy/global_web_traffic_2026.csv')
catalog             = pd.read_csv('Data Sources copy/Streaming Platform Dataset/streaming_catalog_enriched.csv')
platform_summary    = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_summary.csv')
title_financials    = pd.read_csv('Data Sources copy/Streaming Platform Dataset/title_financial_estimates.csv')
industry_metrics    = pd.read_csv('Data Sources copy/Streaming Platform Dataset/industry_metrics.csv')
platform_financials = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_financials_comprehensive.csv')

print('Data sources loaded successfully.')
print(f'  streaming_service:   {streaming_service.shape}')
print(f'  web_traffic:         {web_traffic.shape}')
print(f'  catalog:             {catalog.shape}')
print(f'  platform_summary:    {platform_summary.shape}')
print(f'  title_financials:    {title_financials.shape}')
print(f'  industry_metrics:    {industry_metrics.shape}')
print(f'  platform_financials: {platform_financials.shape}')

Data sources loaded successfully.
  streaming_service:   (777, 3)
  web_traffic:         (1500, 9)
  catalog:             (4, 26)
  platform_summary:    (12, 36)
  title_financials:    (13, 13)
  industry_metrics:    (9, 8)
  platform_financials: (10, 32)


---
## Part 4 – Dimension Design

### Overview: Planned Fact Tables
The constellation schema will contain **three fact tables**, each requiring its own set of dimensions:

| Fact Table | Primary Source(s) | Grain |
|---|---|---|
| `fact_platform_engagement` | `global_web_traffic_2026.csv` | One row per domain per crawl date |
| `fact_platform_financials` | `platform_financials_comprehensive.csv`, `platform_summary.csv` | One row per platform per reporting period |
| `fact_subscription_pricing` | `streaming_service.csv` | One row per service per month |

---
### Dimension Tables

#### DIM_DATE (Date Dimension)
Used by all three fact tables.

| Attribute | Type | Notes |
|---|---|---|
| date_key (PK) | INT | Surrogate key, e.g. 20240101 |
| full_date | DATE | Actual date |
| day_of_week | VARCHAR | Monday–Sunday |
| month_num | INT | 1–12 |
| month_name | VARCHAR | January–December |
| quarter | INT | 1–4 |
| year | INT | e.g. 2024 |
| decade | VARCHAR | e.g. '2010s', '2020s' |
| is_weekend | BOOLEAN | TRUE/FALSE |

**Sources:** `streaming_service.date`, `platform_financials.reporting_date`, `web_traffic.last_crawled`

In [10]:
#Date fields from each source
print('streaming_service: date column')
print(streaming_service['date'].head(5).to_string())

print('platform_financials: reporting_date column')
print(platform_financials['reporting_date'].head(5).to_string())

print('web_traffic: last_crawled column')
print(web_traffic['last_crawled'].head(5).to_string())

streaming_service: date column
0    Jul-2011
1    Aug-2011
2    Sep-2011
3    Oct-2011
4    Nov-2011
platform_financials: reporting_date column
0    2026-01-19
1    2026-02-11
2    2026-02-07
3    2026-02-06
4    2026-02-06
web_traffic: last_crawled column
0    2026-02-07
1    2026-02-07
2    2026-02-07
3    2026-02-07
4    2026-02-07


---
#### DIM_PLATFORM (Platform Dimension) – **Hierarchy Dimension**
Used by `fact_platform_financials` and `fact_platform_engagement`.

This is the **hierarchy dimension** because platforms naturally roll up into parent companies, and parent companies roll up into media sectors.

**Hierarchy:** `Media Sector → Parent Company → Platform → Business Model`

| Attribute | Type | Notes |
|---|---|---|
| platform_key (PK) | INT | Surrogate key |
| platform_name | VARCHAR | e.g. 'Netflix', 'Hulu' |
| parent_company | VARCHAR | e.g. 'Warner Bros. Discovery' |
| media_sector | VARCHAR | 'Streaming', 'Traditional TV', 'Radio', 'Print' |
| content_focus | VARCHAR | e.g. 'General Entertainment', 'Sports', 'Kids' |
| business_model | VARCHAR | 'SVOD', 'AVOD', 'FAST', 'Hybrid' |
| launch_year | INT | Year platform was founded |
| platform_age_years | INT | Calculated from launch_year |
| is_digital | BOOLEAN | TRUE for streaming, FALSE for traditional |

**Sources:** `platform_summary.csv`, `platform_financials_comprehensive.csv`

In [20]:
#platform hierarchy fields
hierarchy_cols = ['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']
available = [c for c in hierarchy_cols if c in platform_summary.columns]
print('DIM_PLATFORM hierarchy fields from platform_summary:')
print(platform_summary[available].to_string())

DIM_PLATFORM hierarchy fields from platform_summary:
              platform           parent_company                                   content_focus               business_model  launch_year
0              Netflix             Netflix Inc.        Original series, films, licensed content           Subscription + Ads         2007
1              Disney+  The Walt Disney Company  Family, Marvel, Star Wars, National Geographic           Subscription + Ads         2019
2                 Hulu  The Walt Disney Company                    TV shows, next-day broadcast           Subscription + Ads         2008
3   Amazon Prime Video          Amazon.com Inc.                 Original series, movies, sports  Prime bundle + Subscription         2006
4           Paramount+         Paramount Global                      Nickelodeon, CBS, Showtime           Subscription + Ads         2021
5              Peacock                  Comcast                            NBCUniversal content           Subscription 

---
#### DIM_GEOGRAPHY (Geography/Market Dimension)
Used by `fact_platform_engagement`.

| Attribute | Type | Notes |
|---|---|---|
| geography_key (PK) | INT | Surrogate key |
| market_code | VARCHAR(2) | ISO country code, e.g. 'US', 'GB' |
| country_name | VARCHAR | e.g. 'United States' |
| region | VARCHAR | e.g. 'North America', 'Europe' |
| subregion | VARCHAR | e.g. 'Western Europe' |

**Source:** `global_web_traffic_2026.primary_market`

In [27]:
# geography values
print('Unique primary_market values')
print(web_traffic['primary_market'].value_counts())

Unique primary_market values
primary_market
JP    265
US    265
BR    255
DE    243
IN    237
GB    235
Name: count, dtype: int64


---
#### DIM_CONTENT_CATEGORY (Content Category Dimension)
Used by `fact_platform_engagement`.

| Attribute | Type | Notes |
|---|---|---|
| category_key (PK) | INT | Surrogate key |
| category | VARCHAR | e.g. 'Streaming', 'News', 'Social', 'Finance' |
| is_media_related | BOOLEAN | TRUE for Streaming/News/Social |
| category_description | VARCHAR | Brief explanation |

**Source:** `global_web_traffic_2026.category`

In [29]:
#category values
print('Unique category values in web_traffic')
print(web_traffic['category'].value_counts())

Unique category values in web_traffic
category
Search        243
E-commerce    216
News          214
Finance       214
Social        207
Streaming     204
Tech          202
Name: count, dtype: int64


---
#### DIM_SUBSCRIPTION_PLAN (Subscription Plan Dimension)
Used by `fact_subscription_pricing`.

| Attribute | Type | Notes |
|---|---|---|
| plan_key (PK) | INT | Surrogate key |
| service_name | VARCHAR | e.g. 'Netflix' |
| price_usd | DECIMAL(6,2) | Monthly price |
| price_tier | VARCHAR | 'Budget' (<$7), 'Mid' ($7–$14), 'Premium' (>$14) |
| effective_date | DATE | When this price took effect |
| end_date | DATE | NULL if currently active |
| is_current | BOOLEAN | SCD Type 2 indicator |

**Source:** `streaming_service.csv`  
> This dimension uses **SCD Type 2** because prices change over time and historical pricing is needed to answer business question #1 (platform growth/decline correlations).

In [34]:
#pricing data
print('streaming_service: first 10 rows')
print(streaming_service.head(10).to_string())

print('Price range per service')
print(streaming_service.groupby('service')['price'].agg(['min', 'max', 'count']))

streaming_service: first 10 rows
   service      date  price
0  Netflix  Jul-2011   7.99
1  Netflix  Aug-2011   7.99
2  Netflix  Sep-2011   7.99
3  Netflix  Oct-2011   7.99
4  Netflix  Nov-2011   7.99
5  Netflix  Dec-2011   7.99
6  Netflix  Jan-2012   7.99
7  Netflix  Feb-2012   7.99
8  Netflix  Mar-2012   7.99
9  Netflix  Apr-2012   7.99
Price range per service
               min    max  count
service                         
Apple TV+     4.99   9.99     51
Crunchyroll   7.99   7.99     42
Disney+       6.99  13.99     51
HBO Max      14.99  15.99     45
Hulu         11.99  17.99    100
Netflix       7.99  15.49    151
Paramount+    9.99  11.99    112
Peacock       9.99  11.99     43
Prime Video   8.99  11.99     94
Shudder       5.99   6.99     88


---
## Slowly Changing Dimensions (SCD)

| Dimension | Attribute | SCD Type | Reason |
|---|---|---|---|
| `DIM_SUBSCRIPTION_PLAN` | `price_usd` | **Type 2** | Historical pricing needed to correlate price hikes with subscriber churn (BQ #1, #5) |
| `DIM_PLATFORM` | `business_model` | **Type 2** | Platforms shift from SVOD to Hybrid/AVOD (e.g. Netflix added ads in 2022); must preserve history |
| `DIM_PLATFORM` | `subscribers_millions` | **Type 1** | Current subscriber count is what matters; overwrite with latest figure |

> **SCD Type 1** – Overwrite: used when only the current value matters (no historical tracking needed).  
> **SCD Type 2** – Add new row: used when historical values are needed for trend analysis.

---
## Data Transformations Required

The following transformations are needed to standardize data across sources before loading into the warehouse.

### Transformation 1 – Date Standardization (`streaming_service.date`)
**Issue:** Dates are stored as abbreviated month-year strings (e.g. `Jul-2011`), not a proper DATE format.  
**Target format:** `YYYY-MM-DD` (first of month assumed).

In [40]:
print('BEFORE transformation')
print(streaming_service[['service', 'date', 'price']].head(5).to_string())

# Transformation
streaming_service['date_transformed'] = pd.to_datetime(
    streaming_service['date'], format='%b-%Y'
)

print('AFTER transformation')
print(streaming_service[['service', 'date', 'date_transformed', 'price']].head(5).to_string())

BEFORE transformation
   service      date  price
0  Netflix  Jul-2011   7.99
1  Netflix  Aug-2011   7.99
2  Netflix  Sep-2011   7.99
3  Netflix  Oct-2011   7.99
4  Netflix  Nov-2011   7.99
AFTER transformation
   service      date date_transformed  price
0  Netflix  Jul-2011       2011-07-01   7.99
1  Netflix  Aug-2011       2011-08-01   7.99
2  Netflix  Sep-2011       2011-09-01   7.99
3  Netflix  Oct-2011       2011-10-01   7.99
4  Netflix  Nov-2011       2011-11-01   7.99


### Transformation 2 – Session Duration Unit Conversion (`web_traffic.avg_session_duration_s`)
**Issue:** Average session duration is stored in **seconds**, making it hard to compare across analysts who think in minutes.  
**Transformation:** Add a derived column `avg_session_duration_min` rounded to 2 decimal places.

In [42]:
print('BEFORE transformation (seconds)')
print(web_traffic[['domain', 'category', 'avg_session_duration_s']].head(8).to_string())

# Transformation: seconds -> minutes
web_traffic['avg_session_duration_min'] = (web_traffic['avg_session_duration_s'] / 60).round(2)

print('AFTER transformation (minutes added)')
print(web_traffic[['domain', 'category', 'avg_session_duration_s', 'avg_session_duration_min']].head(8).to_string())

BEFORE transformation (seconds)
             domain category  avg_session_duration_s
0        google.com   Social                  497.97
1  gtld-servers.net   Social                  281.25
2     microsoft.com   Social                  507.02
3      facebook.com     News                  193.58
4           mail.ru  Finance                  300.91
5    cloudflare.com   Search                   74.71
6    googleapis.com   Social                  389.54
7         apple.com   Social                  303.73
AFTER transformation (minutes added)
             domain category  avg_session_duration_s  avg_session_duration_min
0        google.com   Social                  497.97                      8.30
1  gtld-servers.net   Social                  281.25                      4.69
2     microsoft.com   Social                  507.02                      8.45
3      facebook.com     News                  193.58                      3.23
4           mail.ru  Finance                  300.91       

### Transformation 3 – Price Tier Derivation (`streaming_service.price`)
**Issue:** `price` is a raw numeric field. Business users need a categorical price tier to group platforms for comparison (BQ #5 – factors influencing switch).  
**Transformation:** Bin price into `Budget`, `Mid`, and `Premium` tiers.

In [53]:
print('BEFORE transformation')
print(streaming_service[['service', 'price']].describe())

bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0–$7)', 'Mid ($7–$14)', 'Premium (>$14)']
streaming_service['price_tier'] = pd.cut(
    streaming_service['price'], bins=bins, labels=labels, right=True
)

print('AFTER transformation (price_tier added)')
print(streaming_service[['service', 'date_transformed', 'price', 'price_tier']].head(10).to_string())

BEFORE transformation
            price
count  777.000000
mean     9.378674
std      2.609859
min      4.990000
25%      7.990000
50%      8.990000
75%      9.990000
max     17.990000
AFTER transformation (price_tier added)
   service date_transformed  price    price_tier
0  Netflix       2011-07-01   7.99  Mid ($7–$14)
1  Netflix       2011-08-01   7.99  Mid ($7–$14)
2  Netflix       2011-09-01   7.99  Mid ($7–$14)
3  Netflix       2011-10-01   7.99  Mid ($7–$14)
4  Netflix       2011-11-01   7.99  Mid ($7–$14)
5  Netflix       2011-12-01   7.99  Mid ($7–$14)
6  Netflix       2012-01-01   7.99  Mid ($7–$14)
7  Netflix       2012-02-01   7.99  Mid ($7–$14)
8  Netflix       2012-03-01   7.99  Mid ($7–$14)
9  Netflix       2012-04-01   7.99  Mid ($7–$14)


In [51]:
print('Tier distribution')
print(streaming_service['price_tier'].value_counts())

Tier distribution
price_tier
Mid ($7–$14)      560
Budget ($0–$7)    154
Premium (>$14)     63
Name: count, dtype: int64


---
## Summary: Dimension Table Assignments

| Dimension Table | Type | Fact Tables Used In | SCD | Hierarchy? |
|---|---|---|---|---|
| `DIM_DATE` | Date | All three | N/A | No |
| `DIM_PLATFORM` | Non-date | `fact_platform_financials`, `fact_platform_engagement` | Type 1 (subscribers), Type 2 (business_model) | **Yes** – Sector → Company → Platform |
| `DIM_GEOGRAPHY` | Non-date | `fact_platform_engagement` | Type 1 | No |
| `DIM_CONTENT_CATEGORY` | Non-date | `fact_platform_engagement` | Type 1 | No |
| `DIM_SUBSCRIPTION_PLAN` | Non-date | `fact_subscription_pricing` | **Type 2** (price history) | No |

> Each fact table has **at least one date dimension** and **at least two non-date dimensions**, satisfying the project requirements.

---
---
# Part 5 – Facts

Reviewing each data source against the five business questions to identify **raw measurements** (stored directly in fact tables) and **calculated measures** (derived at query time via aggregate functions or transformations).

---
### Fact Table 1: `fact_platform_engagement`
**Source:** `global_web_traffic_2026.csv`  
**Grain:** One row per domain per crawl date  
**Answers:** BQ #1 (which platforms show most engagement), BQ #4 (engagement differences between digital and traditional)

#### Raw Measurements
| Column | Additive Type | Notes |
|---|---|---|
| `monthly_visits` | Fully additive | Can be summed across domains and markets |
| `avg_session_duration_s` | Semi-additive | Average — meaningful per domain, not summed across domains |
| `bounce_rate_pct` | Semi-additive | Must be averaged, not summed |
| `stickiness_index` | Semi-additive | Composite score — compare via average only |
| `global_rank` | Non-additive | Ordinal rank — never summed; snapshot only |

#### Calculated Measures
| Measure | Formula | Business Question |
|---|---|---|
| Avg session duration (min) | `avg_session_duration_s / 60` | BQ #4 – engagement comparison |
| Streaming share of total visits | `SUM(visits WHERE category='Streaming') / SUM(all visits)` | BQ #1 – digital vs. traditional share |
| Engagement rank within category | `RANK() OVER (PARTITION BY category ORDER BY stickiness_index DESC)` | BQ #1 – top platforms per category |
| YoY visit growth (%) | `(current_visits − prior_year_visits) / prior_year_visits * 100` | BQ #2 – when did shift accelerate |

In [ ]:
import pandas as pd

web_traffic = pd.read_csv('Data Sources copy/global_web_traffic_2026.csv')

print('=== fact_platform_engagement: raw measurement columns ===')
fact_eng_cols = ['monthly_visits', 'avg_session_duration_s', 'bounce_rate_pct', 'stickiness_index', 'global_rank']
print(web_traffic[fact_eng_cols].describe().round(2))

print('\n=== Calculated measure: streaming share of total visits ===')
total_visits = web_traffic['monthly_visits'].sum()
by_cat = web_traffic.groupby('category')['monthly_visits'].sum().reset_index()
by_cat['visit_share_pct'] = (by_cat['monthly_visits'] / total_visits * 100).round(2)
by_cat = by_cat.sort_values('monthly_visits', ascending=False)
print(by_cat.to_string(index=False))

In [ ]:
# Calculated measure: engagement rank within category
web_traffic['engagement_rank'] = web_traffic.groupby('category')['stickiness_index'] \
                                             .rank(ascending=False, method='dense')

print('=== Calculated measure: top 5 Streaming domains by stickiness ===')
streaming = web_traffic[web_traffic['category'] == 'Streaming'] \
              .sort_values('stickiness_index', ascending=False) \
              .head(5)
print(streaming[['domain', 'monthly_visits', 'avg_session_duration_s',
                  'bounce_rate_pct', 'stickiness_index', 'engagement_rank']].to_string(index=False))

---
### Fact Table 2: `fact_platform_financials`
**Sources:** `platform_financials_comprehensive.csv`, `platform_summary.csv`  
**Grain:** One row per platform per reporting period  
**Answers:** BQ #1 (platform growth/decline), BQ #2 (when shift accelerated), BQ #5 (factors driving switch)

#### Raw Measurements
| Column | Additive Type | Notes |
|---|---|---|
| `quarterly_revenue_usd_millions` | Fully additive | Sums across platforms and quarters |
| `quarterly_profit_usd_millions` | Fully additive | Sums across platforms and quarters |
| `subscribers_millions` | Semi-additive | Can sum across platforms for total market; not across time periods |
| `annual_content_spend_usd_millions` | Fully additive | Total content investment per platform per year |
| `ad_revenue_2025_usd_millions` | Fully additive | Ad revenue per platform |
| `streaming_hours_billions` | Fully additive | Hours watched; additive across platforms and periods |
| `sports_rights_spend_usd_millions` | Fully additive | Sports rights investment per platform |

#### Calculated Measures
| Measure | Formula | Business Question |
|---|---|---|
| Profit margin (%) | `quarterly_profit / quarterly_revenue * 100` | BQ #1 – platform financial health |
| Revenue per subscriber | `quarterly_revenue / subscribers_millions` | BQ #5 – monetization comparison |
| Content spend as % of revenue | `annual_content_spend / (quarterly_revenue * 4) * 100` | BQ #5 – investment intensity |
| Subscriber growth rate YoY | `(current_subs − prior_year_subs) / prior_year_subs * 100` | BQ #2 – shift acceleration |
| Ad revenue as % of total revenue | `ad_revenue / (quarterly_revenue * 4) * 100` | BQ #1 – business model mix |

In [ ]:
platform_financials = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_financials_comprehensive.csv')

print('=== fact_platform_financials: raw measurements ===')
fact_fin_cols = [
    'platform', 'subscribers_millions', 'quarterly_revenue_usd_millions',
    'quarterly_profit_usd_millions', 'annual_content_spend_usd_millions',
    'ad_revenue_2025_usd_millions', 'streaming_hours_2025_billions'
]
available = [c for c in fact_fin_cols if c in platform_financials.columns]
print(platform_financials[available].to_string(index=False))

# Calculated measures: profit margin and revenue per subscriber
pf = platform_financials.copy()
pf['profit_margin_pct'] = (
    pd.to_numeric(pf['quarterly_profit_usd_millions'], errors='coerce') /
    pd.to_numeric(pf['quarterly_revenue_usd_millions'], errors='coerce') * 100
).round(2)

pf['revenue_per_subscriber'] = (
    pd.to_numeric(pf['quarterly_revenue_usd_millions'], errors='coerce') /
    pd.to_numeric(pf['subscribers_millions'], errors='coerce')
).round(2)

print('\n=== Calculated measures: profit margin & revenue per subscriber ===')
calc_cols = ['platform', 'profit_margin_pct', 'revenue_per_subscriber']
available_calc = [c for c in calc_cols if c in pf.columns]
print(pf[available_calc].dropna(how='all', subset=available_calc[1:]).to_string(index=False))

---
### Fact Table 3: `fact_subscription_pricing`
**Source:** `streaming_service.csv`  
**Grain:** One row per service per month  
**Answers:** BQ #5 (factors influencing platform switch), BQ #1 (platform competitiveness over time)

#### Raw Measurements
| Column | Additive Type | Notes |
|---|---|---|
| `price` | Non-additive | Point-in-time price — summing prices across services is meaningless; must compare |

#### Calculated Measures
| Measure | Formula | Business Question |
|---|---|---|
| Month-over-month price change ($) | `price − LAG(price,1) OVER (PARTITION BY service ORDER BY date)` | BQ #5 – what triggers churn |
| Cumulative price increase since launch | `price − FIRST_VALUE(price) OVER (PARTITION BY service ORDER BY date)` | BQ #1 – long-term pricing trend |
| Price index relative to Netflix | `service_price / netflix_price_same_month * 100` | BQ #5 – competitive positioning |
| Average industry price per month | `AVG(price) GROUP BY date` | BQ #2 – market-wide pricing trends |

In [ ]:
streaming_service = pd.read_csv('Data Sources copy/streaming_service.csv')
streaming_service['date'] = pd.to_datetime(streaming_service['date'], format='%b-%Y')
streaming_service = streaming_service.sort_values(['service', 'date'])

# Calculated measures: MoM price change and cumulative increase
streaming_service['price_change_mom'] = streaming_service.groupby('service')['price'].diff()
streaming_service['cumulative_increase'] = streaming_service.groupby('service')['price'] \
    .transform(lambda x: x - x.iloc[0])

print('=== Rows where price changed (MoM price change != 0) ===')
price_changes = streaming_service[streaming_service['price_change_mom'] != 0].dropna(subset=['price_change_mom'])
print(price_changes[['service', 'date', 'price', 'price_change_mom', 'cumulative_increase']].to_string(index=False))

# Calculated measure: average industry price per month
avg_price_by_month = streaming_service.groupby('date')['price'].mean().reset_index()
avg_price_by_month.columns = ['date', 'avg_industry_price']
avg_price_by_month['avg_industry_price'] = avg_price_by_month['avg_industry_price'].round(2)

print('\n=== Average industry subscription price by month (last 12 months) ===')
print(avg_price_by_month.tail(12).to_string(index=False))

---
## Part 5 Summary: All Facts and Measures

### Raw Measurements by Fact Table

| Fact Table | Measurement | Additive Type |
|---|---|---|
| `fact_platform_engagement` | monthly_visits | Fully additive |
| | avg_session_duration_s | Semi-additive |
| | bounce_rate_pct | Semi-additive |
| | stickiness_index | Semi-additive |
| | global_rank | Non-additive |
| `fact_platform_financials` | quarterly_revenue_usd_millions | Fully additive |
| | quarterly_profit_usd_millions | Fully additive |
| | subscribers_millions | Semi-additive |
| | annual_content_spend_usd_millions | Fully additive |
| | ad_revenue_2025_usd_millions | Fully additive |
| | streaming_hours_billions | Fully additive |
| | sports_rights_spend_usd_millions | Fully additive |
| `fact_subscription_pricing` | price | Non-additive |

### Calculated Measures by Fact Table

| Fact Table | Calculated Measure | Transformation Used |
|---|---|---|
| `fact_platform_engagement` | Avg session duration (min) | Arithmetic: `/ 60` |
| | Streaming share of total visits | `SUM() / SUM()` |
| | Engagement rank within category | `RANK() OVER (PARTITION BY category)` |
| | YoY visit growth (%) | `LAG()` window function |
| `fact_platform_financials` | Profit margin (%) | `profit / revenue * 100` |
| | Revenue per subscriber | `revenue / subscribers` |
| | Content spend as % of revenue | `spend / revenue * 100` |
| | Subscriber growth YoY | `LAG() OVER (PARTITION BY platform)` |
| | Ad revenue as % of total revenue | `ad_revenue / revenue * 100` |
| `fact_subscription_pricing` | MoM price change ($) | `LAG(price, 1) OVER (PARTITION BY service)` |
| | Cumulative price increase | `FIRST_VALUE() OVER (PARTITION BY service)` |
| | Average industry price | `AVG(price) GROUP BY date` |
| | Price index vs. Netflix | `service_price / netflix_price * 100` |